In [2]:
# -*- coding: utf-8 -*-
"""
prepare_fashion200k_splits.py

CHẠY FILE NÀY ĐÚNG 1 LẦN DUY NHẤT (ở máy/notebook có internet ổn định).
Mục đích: tải 1000 mẫu Fashion200k, tách 70/15/15 (seed cố định), đóng gói
mỗi split thành 1 file .zip độc lập (ảnh + metadata.json) để bạn:
  1. Tải 3 file .zip này về máy
  2. Upload lên GitHub (hoặc Kaggle Dataset / Google Drive)
  3. Từ nay về sau, File1/File2/File3 (và các file tuần sau) chỉ cần TẢI LẠI
     đúng 3 file này -> không phải stream + shuffle + split lại từ đầu mỗi
     lần chạy -> đảm bảo MỌI notebook trong suốt project dùng chung 1 bộ
     train/val/test giống hệt nhau, so sánh công bằng giữa các tuần.

Output: fashion200k_train.zip, fashion200k_val.zip, fashion200k_test.zip
"""

!pip install -q datasets Pillow

import os, json, random, shutil
from datasets import load_dataset
from PIL import Image

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

N_SAMPLES  = 1000
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# phần còn lại (0.15) là TEST

WORK_DIR = "fashion200k_prep"
os.makedirs(WORK_DIR, exist_ok=True)

print(f"Đang tải {N_SAMPLES} mẫu từ Marqo/fashion200k (streaming)...")
dataset = load_dataset("Marqo/fashion200k", split="data", streaming=True)
samples = list(dataset.take(N_SAMPLES))

raw_img_dir = os.path.join(WORK_DIR, "all_images")
os.makedirs(raw_img_dir, exist_ok=True)

metadata = []
for idx, sample in enumerate(samples):
    caption = (sample.get("text") or "").strip()
    if not caption:
        continue  # bỏ mẫu thiếu caption

    image_file = f"img_{idx:04d}.jpg"
    img = sample["image"]
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.save(os.path.join(raw_img_dir, image_file), quality=95)

    metadata.append({
        "id": idx,
        "item_id": sample.get("item_ID", idx),
        "image_file": image_file,      # đường dẫn TƯƠNG ĐỐI trong thư mục images/
        "caption": caption,
        "category1": sample.get("category1", ""),
        "category2": sample.get("category2", ""),
        "category3": sample.get("category3", ""),
    })

print(f"Đã tải hợp lệ {len(metadata)} mẫu")

# Tách train/val/test — seed cố định, dùng CHUNG cho mọi file/tuần về sau
indices = list(range(len(metadata)))
random.Random(RANDOM_SEED).shuffle(indices)
n = len(indices)
train_end = int(n * TRAIN_FRAC)
val_end   = int(n * (TRAIN_FRAC + VAL_FRAC))

splits = {
    "train": [metadata[i] for i in indices[:train_end]],
    "val":   [metadata[i] for i in indices[train_end:val_end]],
    "test":  [metadata[i] for i in indices[val_end:]],
}

for split_name, split_data in splits.items():
    split_dir = os.path.join(WORK_DIR, split_name)
    img_dir = os.path.join(split_dir, "images")
    os.makedirs(img_dir, exist_ok=True)

    for item in split_data:
        shutil.copy(
            os.path.join(raw_img_dir, item["image_file"]),
            os.path.join(img_dir, item["image_file"]),
        )

    with open(os.path.join(split_dir, "metadata.json"), "w", encoding="utf-8") as f:
        json.dump(split_data, f, ensure_ascii=False, indent=2)

    zip_base = f"fashion200k_{split_name}"
    shutil.make_archive(zip_base, "zip", split_dir)
    size_mb = os.path.getsize(f"{zip_base}.zip") / 1e6
    print(f"{split_name:5s}: {len(split_data):4d} mẫu -> {zip_base}.zip ({size_mb:.1f} MB)")

print("\n✅ Xong. 3 file zip nằm ở thư mục làm việc hiện tại:")
print("   fashion200k_train.zip")
print("   fashion200k_val.zip")
print("   fashion200k_test.zip")
print("\n⚠️ Lưu ý khi upload GitHub:")
print("  - GitHub chặn cứng file > 100MB nếu không dùng Git LFS, và cảnh báo")
print("    từ 50MB. Nếu fashion200k_train.zip vượt ngưỡng này, giảm N_SAMPLES,")
print("    giảm quality khi lưu ảnh (vd quality=85), hoặc dùng Kaggle Dataset/")
print("    Google Drive riêng cho phần ảnh thay vì GitHub.")
print("  - fashion_test.zip CHỈ dùng đúng 1 lần ở cuối kỳ (đánh giá cuối cùng),")
print("    KHÔNG dùng để so sánh phương pháp giữa các tuần.")

Đang tải 1000 mẫu từ Marqo/fashion200k (streaming)...
Đã tải hợp lệ 1000 mẫu
train:  700 mẫu -> fashion200k_train.zip (24.7 MB)
val  :  150 mẫu -> fashion200k_val.zip (5.8 MB)
test :  150 mẫu -> fashion200k_test.zip (5.7 MB)

✅ Xong. 3 file zip nằm ở thư mục làm việc hiện tại:
   fashion200k_train.zip
   fashion200k_val.zip
   fashion200k_test.zip

⚠️ Lưu ý khi upload GitHub:
  - GitHub chặn cứng file > 100MB nếu không dùng Git LFS, và cảnh báo
    từ 50MB. Nếu fashion200k_train.zip vượt ngưỡng này, giảm N_SAMPLES,
    giảm quality khi lưu ảnh (vd quality=85), hoặc dùng Kaggle Dataset/
    Google Drive riêng cho phần ảnh thay vì GitHub.
  - fashion_test.zip CHỈ dùng đúng 1 lần ở cuối kỳ (đánh giá cuối cùng),
    KHÔNG dùng để so sánh phương pháp giữa các tuần.
